# 03. Small-model SFT: from requests to reliable JSON

A companion to the [post-training primer](https://www.norange.ai/post-training/2026/08/24/post-training-primer.html).

Fine-tune **Qwen2.5-0.5B (the base model)** on **256 synthetic demonstrations** using LoRA. The task is deliberately small: read an order request and return its item, color, and quantity as JSON. All information is in the request; we are practicing extraction and a response format, not teaching new world knowledge.

Choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order. The configuration targets a few minutes of training on a Colab GPU (96 optimizer steps); downloading the model and evaluating it take additional time. Runtime and improvement are not pre-measured or guaranteed. No API key, paid model, external dataset, or training service is needed.

We compare three conditions on the **same 64 held-out requests**:

1. Base model, instruction only.
2. Base model, instruction plus four training demonstrations in the prompt (no weight updates).
3. SFT model, instruction only—the identical prompt and decoding settings used in condition 1.

The few-shot control matters: if the base model succeeds with examples, the skill is accessible before fine-tuning. SFT may make it accessible with a shorter prompt. This is evidence consistent with elicitation, not proof that SFT learned nothing new.


In [ ]:
%pip -q install "transformers==4.57.1" "peft==0.17.1" "accelerate==1.10.1" pandas matplotlib tqdm


## Setup

We keep Colab's installed PyTorch. If packages were already imported before installation, restart the runtime once and run from the top. A GPU is required for the intended runtime.


In [ ]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, set_seed
from peft import LoraConfig, TaskType, get_peft_model

SEED = 7
MODEL_ID = "Qwen/Qwen2.5-0.5B"
TRAIN_SIZE = 256
TEST_SIZE = 64
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 3e-4
MAX_LENGTH = 192
MAX_NEW_TOKENS = 64

set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU in Runtime → Change runtime type, then rerun.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print("Training updates:", math.ceil(TRAIN_SIZE / BATCH_SIZE) * EPOCHS)


## A small dataset with an explicit held-out split

Training and test requests have **disjoint (item, color, quantity) combinations**. All individual values occur in training. Half of the test set uses familiar sentence templates; half uses two sentence templates never used for training. We report these groups separately.

The test set is fixed before training. We do not select a checkpoint, learning rate, or examples based on its results. This is a narrow synthetic task, not a broad language-understanding benchmark.


In [ ]:
COLORS = ["red", "blue", "green", "black", "white", "yellow", "pink", "purple"]
ITEMS = ["mug", "notebook", "bottle", "lamp", "plate", "scarf", "wallet", "backpack"]
TRAIN_TEMPLATES = [
    "Please order {quantity} units of the {color} {item}.",
    "I want a {color} {item}. Quantity: {quantity}.",
    "Add to my order: {item}, color {color}, quantity {quantity}.",
    "Can you send {quantity} of the {item} in {color}?",
]
UNSEEN_TEMPLATES = [
    "The {item} should be {color}; send me {quantity} in total.",
    "For this purchase, choose {color} for the {item} and set the count to {quantity}.",
]
INSTRUCTION = (
    'Extract the order from the request. Return exactly one JSON object with keys '
    '"item", "color", and "quantity". Use lowercase strings and an integer quantity. '
    'Do not include explanations or markdown.\n'
)

rng = random.Random(SEED)
combinations = list(itertools.product(ITEMS, COLORS, range(1, 10)))
rng.shuffle(combinations)

def make_record(values, template, group):
    item, color, quantity = values
    target = dict(item=item, color=color, quantity=quantity)
    return dict(request=template.format(**target), target=target, group=group)

train_records = [
    make_record(values, TRAIN_TEMPLATES[i % len(TRAIN_TEMPLATES)], "train")
    for i, values in enumerate(combinations[:TRAIN_SIZE])
]
test_records = []
for i, values in enumerate(combinations[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]):
    familiar = i < TEST_SIZE // 2
    templates = TRAIN_TEMPLATES if familiar else UNSEEN_TEMPLATES
    test_records.append(make_record(
        values, templates[i % len(templates)],
        "New combinations" if familiar else "New wording",
    ))

# Four fixed training examples, one from each familiar template.
few_shot_examples = train_records[:4]

def target_text(record):
    return json.dumps(record["target"], ensure_ascii=False)

def prompt_for(record, examples=()):
    demonstrations = "".join(
        f'Request: {example["request"]}\nJSON: {target_text(example)}\n\n'
        for example in examples
    )
    return INSTRUCTION + demonstrations + f'Request: {record["request"]}\nJSON: '

def record_key(record):
    return tuple(record["target"][key] for key in ("item", "color", "quantity"))

assert len(train_records) == TRAIN_SIZE and len(test_records) == TEST_SIZE
assert not {record_key(r) for r in train_records} & {record_key(r) for r in test_records}
assert not {r["request"] for r in train_records} & {r["request"] for r in test_records}
for key in ("item", "color", "quantity"):
    assert {r["target"][key] for r in test_records} <= {r["target"][key] for r in train_records}
display(pd.DataFrame([
    {"request": r["request"], "target": target_text(r)} for r in train_records[:4]
]))


## Load the pretrained model

We use the **base checkpoint**, not an instruction-tuned checkpoint. Plain-text prompts are intentional; no chat template is applied. Generation uses greedy decoding, no constrained JSON grammar, and the same token budget in every condition.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
# Left padding is needed for batched decoder-only generation.
tokenizer.padding_side = "left"
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, attn_implementation="sdpa",
).to(DEVICE)
base_model.eval()

generation_config = GenerationConfig(
    do_sample=False,
    max_new_tokens=MAX_NEW_TOKENS,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    use_cache=True,
)

@torch.inference_mode()
def generate_responses(model, records, examples=()):
    model.eval()
    outputs = []
    for start in tqdm(range(0, len(records), EVAL_BATCH_SIZE), desc="Generate"):
        batch = records[start:start + EVAL_BATCH_SIZE]
        prompts = [prompt_for(record, examples) for record in batch]
        inputs = tokenizer(prompts, padding=True, return_tensors="pt",
                           add_special_tokens=False).to(DEVICE)
        generated = model.generate(**inputs, generation_config=generation_config)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return outputs


## Measure format and content separately

**JSON-only rate** requires the complete output to be a JSON object (surrounding whitespace is allowed). **Correct content** checks the first JSON object even when surrounded by extra prose. **Exact task success** requires JSON-only output, exactly the three requested keys, correct values, and an integer quantity. Key order and spacing do not matter.

This distinction helps reveal a formatting improvement rather than calling every malformed response a failure to understand the request. All three conditions use the same scoring rules. We also retain every raw response for inspection.


In [ ]:
def score_response(text, target):
    try:
        strict = json.loads(text.strip())
    except (json.JSONDecodeError, ValueError):
        strict = None
    json_only = isinstance(strict, dict)
    # For the content diagnostic, inspect only the first decodable object.
    parsed = strict if json_only else None
    if parsed is None:
        for i, character in enumerate(text):
            if character != "{":
                continue
            try:
                candidate, _ = json.JSONDecoder().raw_decode(text[i:])
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict):
                parsed = candidate
                break
    parsed = parsed if isinstance(parsed, dict) else {}
    content = all(
        type(parsed.get(key)) is type(value) and parsed.get(key) == value
        for key, value in target.items()
    )
    return {
        "JSON only": json_only,
        "Correct content": content,
        "Exact task success": json_only and content and set(parsed) == set(target),
    }

def evaluate(outputs, records, condition):
    return pd.DataFrame([
        {"condition": condition, "group": record["group"], "request": record["request"],
         "target": target_text(record), "output": output,
         **score_response(output, record["target"])}
        for record, output in zip(records, outputs, strict=True)
    ])

# Sanity checks ensure that prose and incorrect types are not counted as exact success.
toy = {"item": "mug", "color": "red", "quantity": 2}
assert score_response(json.dumps(toy), toy)["Exact task success"]
assert not score_response("Here: " + json.dumps(toy), toy)["Exact task success"]
assert score_response("Here: " + json.dumps(toy), toy)["Correct content"]
assert not score_response('{"item":"mug","color":"red","quantity":"2"}', toy)["Correct content"]
assert not score_response("No JSON here", toy)["JSON only"]

before_outputs = generate_responses(base_model, test_records)
few_shot_outputs = generate_responses(base_model, test_records, few_shot_examples)
before = evaluate(before_outputs, test_records, "Base: instruction only")
few_shot = evaluate(few_shot_outputs, test_records, "Base: four demonstrations")
metrics = ["JSON only", "Correct content", "Exact task success"]
display(pd.concat([before, few_shot]).groupby("condition", sort=False)[metrics].mean().mul(100).round(1))


## SFT: predict demonstrated output tokens

For each training request, the demonstrated JSON is the target. We minimize

$$\mathcal L_{\mathrm{SFT}}=-\frac{1}{N_{\mathrm{target}}}\sum_{i,t\in\mathrm{target}}\log\pi_\theta(y_{i,t}\mid x_i,y_{i,<t}).$$

The prompt and padding positions have label `-100`, so they contribute no loss. Earlier target tokens are supplied by the demonstration (**teacher forcing**). The model shifts labels internally; we must not shift them again. We include the end-of-sequence token so it learns to stop after the answer.

LoRA trains small low-rank adapters in the attention and MLP projections while keeping the original weights frozen. It changes which parameters are optimized, not the SFT objective.


In [ ]:
def encode_record(record):
    # Concatenate token sequences explicitly: the prompt prefix exactly matches inference.
    prefix = tokenizer.encode(prompt_for(record), add_special_tokens=False)
    answer = tokenizer.encode(target_text(record), add_special_tokens=False)
    answer += [tokenizer.eos_token_id]
    input_ids = prefix + answer
    assert len(input_ids) <= MAX_LENGTH, "Raise MAX_LENGTH; do not silently cut off targets."
    return {"input_ids": input_ids, "labels": [-100] * len(prefix) + answer}

encoded_train = [encode_record(record) for record in train_records]

def collate_training(rows):
    # Right padding for training; keep the tokenizer itself left-padded for generation.
    width = max(len(row["input_ids"]) for row in rows)
    ids = torch.full((len(rows), width), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    attention = torch.zeros_like(ids)
    for i, row in enumerate(rows):
        size = len(row["input_ids"])
        ids[i, :size] = torch.tensor(row["input_ids"])
        labels[i, :size] = torch.tensor(row["labels"])
        attention[i, :size] = 1
    return {"input_ids": ids, "labels": labels, "attention_mask": attention}

loader = DataLoader(
    encoded_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_training,
    generator=torch.Generator().manual_seed(SEED),
)
set_seed(SEED)
model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
))
model.print_trainable_parameters()

example = encoded_train[0]
print("Supervised text:", tokenizer.decode([x for x in example["labels"] if x != -100]))
print("Longest training sequence:", max(len(x["input_ids"]) for x in encoded_train))


## Fine-tune for three passes over the data

This is a small, explicit PyTorch loop: forward pass, cross-entropy, backpropagation, optimizer step. No rewards or advantage estimates are used. The learning rate decays linearly after a short warmup. We log the actual wall-clock time and training loss; the held-out test set never enters this loop.

To repeat the experiment from scratch, rerun from **Load the pretrained model**. Rerunning only this cell continues training the existing adapter.


In [ ]:
trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")
total_steps = EPOCHS * len(loader)
warmup_steps = max(1, int(0.1 * total_steps))

def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / (total_steps - warmup_steps))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
model.config.use_cache = False
model.train()
loss_history = []
torch.cuda.synchronize()
started = time.perf_counter()
progress = tqdm(total=total_steps, desc="SFT updates")
for epoch in range(EPOCHS):
    for cpu_batch in loader:
        batch = {key: value.to(DEVICE) for key, value in cpu_batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = model(**batch).loss
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite loss; restart the run with a lower learning rate.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= previous_scale:
            scheduler.step()
        loss_history.append({"update": len(loss_history) + 1,
                             "epoch": epoch + 1, "loss": loss.item()})
        progress.update(1)
        progress.set_postfix(loss=f"{loss.item():.3f}")
progress.close()
torch.cuda.synchronize()
training_seconds = time.perf_counter() - started
model.config.use_cache = True
model.eval()
print(f"Training: {training_seconds / 60:.2f} minutes ({len(loss_history)} batches).")

history = pd.DataFrame(loss_history)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history["update"], history["loss"], color="#a68ac2", alpha=0.4, label="Batch loss")
ax.plot(history["update"], history["loss"].rolling(8, min_periods=1).mean(),
        color="#8764a4", label="8-batch moving average")
ax.set(xlabel="Optimizer batch", ylabel="Target-token cross-entropy", title="SFT training loss")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## Before vs. after: same held-out requests, same decoding

The main comparison is **Base: instruction only** versus **SFT: instruction only**. The few-shot baseline has a longer prompt and is a separate control, not the before condition. Report both familiar and unseen wording; high training performance alone is not evidence of generalization.


In [ ]:
after_outputs = generate_responses(model, test_records)
after = evaluate(after_outputs, test_records, "SFT: instruction only")
results = pd.concat([before, few_shot, after], ignore_index=True)
summary = results.groupby("condition", sort=False)[metrics].mean().mul(100)
display(summary.round(1).style.format("{:.1f}%"))
display(results.groupby(["condition", "group"], sort=False)[metrics].mean().mul(100).round(1))

colors = ["#b8a3ce", "#d9be8e", "#95b7a4"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, metric in zip(axes, ["Correct content", "Exact task success"]):
    values = summary[metric].to_numpy()
    bars = ax.bar(["Base", "Base + 4 examples", "SFT"], values, color=colors)
    ax.bar_label(bars, labels=[f"{value:.1f}%" for value in values], padding=3)
    ax.set(title=metric, ylabel="Held-out requests (%)", ylim=(0, 110))
    ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

before_correct = before["Exact task success"].to_numpy(dtype=bool)
after_correct = after["Exact task success"].to_numpy(dtype=bool)
fixed = int((~before_correct & after_correct).sum())
regressed = int((before_correct & ~after_correct).sum())
delta = 100 * (after_correct.mean() - before_correct.mean())
print(f"Exact task success changed by {delta:+.1f} percentage points.")
print(f"Improved requests: {fixed}/{TEST_SIZE}; regressions: {regressed}/{TEST_SIZE}.")
if delta <= 0:
    print("This run does not demonstrate an improvement. Inspect all outputs below; do not infer a gain from training loss alone.")


## Read the actual outputs

The first examples from each test group are selected **before seeing scores**, so this view includes successes and failures rather than just favorable cases. The full table is saved below.


In [ ]:
example_indices = [0, 1, 2, TEST_SIZE // 2, TEST_SIZE // 2 + 1, TEST_SIZE // 2 + 2]
comparison = pd.DataFrame([
    {"group": record["group"], "request": record["request"], "target": target_text(record),
     "before": before_outputs[i], "base + examples": few_shot_outputs[i], "after": after_outputs[i]}
    for i, record in enumerate(test_records)
])
with pd.option_context("display.max_colwidth", None):
    display(comparison.iloc[example_indices].style.set_properties(**{
        "white-space": "pre-wrap", "text-align": "left", "vertical-align": "top",
    }))

output_dir = Path("sft_order_results")
output_dir.mkdir(exist_ok=True)
comparison.to_csv(output_dir / "all_predictions.csv", index=False)
results.to_csv(output_dir / "scores.csv", index=False)
history.to_csv(output_dir / "training_loss.csv", index=False)
(output_dir / "run.json").write_text(json.dumps({
    "model": MODEL_ID, "model_revision": getattr(model.config, "_commit_hash", None),
    "seed": SEED, "train_size": TRAIN_SIZE, "test_size": TEST_SIZE,
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "training_seconds": training_seconds,
}, indent=2))
print("Saved results in", output_dir.resolve())


## What would support an elicitation interpretation?

- **Base + demonstrations already works:** the pretrained model can perform the task when the prompt supplies examples. This is direct evidence of an accessible capability before weight updates.
- **SFT improves instruction-only performance:** training makes the behavior more reliable without carrying demonstrations in every prompt.
- **Content was correct but formatting improves:** part of the gain is learning the response convention. Compare content and exact-task metrics rather than only one score.
- **New wording also improves:** the model is doing more than repeating exact training requests. The test remains narrow: the vocabulary and schema are shared with training.

If the few-shot base model also fails, this run provides weaker evidence for elicitation. If SFT improves, it may have learned task-specific mappings or procedures. Neither outcome settles whether SFT generally creates new capabilities, and this notebook makes no comparison with RL. A larger, multi-seed study would be needed for a broader claim.

### Sources

- [Qwen2.5-0.5B model card](https://huggingface.co/Qwen/Qwen2.5-0.5B).
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685).
- [LIMA: Less Is More for Alignment](https://arxiv.org/abs/2305.11206), for the elicitation motivation; this small synthetic experiment does not reproduce LIMA.
- [PEFT documentation](https://huggingface.co/docs/peft/en/quicktour).
